In [ ]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface
import spikeinterface.exporters as sexp
from spikeinterface.core import write_binary_recording
from pathlib import Path


In [ ]:
file_dict = {
    'mouse1': {
        1215: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse1_natima_251215_232216',
        1217: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse1_natima_251217_225054',
        1219: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse1&oldV1_natima_251219_201746'
    },
    'mouse2': {
        1214: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse&V1B_natima_251214_154409',
        1215: '/media/ubuntu/sda/mouse_test/raw_data/WLF_V1left&128ch2mouse_natima_251215_223556',
        1216: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse&V1left_natima_251216_214224',
        1217: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1left_natima_251217_220244',
        1218: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1od_natima_251218_214009',
        1219: '/media/ubuntu/sda/mouse_test/raw_data/WLF_128ch2mouse2&V1left_natima_251219_192148'
    }

}

In [ ]:
# 批量处理循环 - 遍历file_dict中的所有数据
# file_list将在循环中根据每个数据路径动态生成

In [ ]:
channel_list_A = ['A-000', 'A-001', 'A-002',
       'A-003', 'A-004', 'A-005', 'A-006', 'A-007', 'A-008', 'A-009',
       'A-010', 'A-011', 'A-012', 'A-013', 'A-014', 'A-015', 'A-016',
       'A-017', 'A-018', 'A-019', 'A-020', 'A-021', 'A-022', 'A-023',
       'A-024', 'A-025', 'A-026', 'A-027', 'A-028', 'A-029', 'A-030',
       'A-031', 'A-032', 'A-033', 'A-034', 'A-035', 'A-036', 'A-037',
       'A-038', 'A-039', 'A-040', 'A-041', 'A-042', 'A-043', 'A-044',
       'A-045', 'A-046', 'A-047', 'A-048', 'A-049', 'A-050', 'A-051',
       'A-052', 'A-053', 'A-054', 'A-055', 'A-056', 'A-057', 'A-058',
       'A-059', 'A-060', 'A-061', 'A-062', 'A-063', 'A-064', 'A-065',
       'A-066', 'A-067', 'A-068', 'A-069', 'A-070', 'A-071', 'A-072',
       'A-073', 'A-074', 'A-075', 'A-076', 'A-077', 'A-078', 'A-079',
       'A-080', 'A-081', 'A-082', 'A-083', 'A-084', 'A-085', 'A-086',
       'A-087', 'A-088', 'A-089', 'A-090', 'A-091', 'A-092', 'A-093',
       'A-094', 'A-095', 'A-096', 'A-097', 'A-098', 'A-099', 'A-100',
       'A-101', 'A-102', 'A-103', 'A-104', 'A-105', 'A-106', 'A-107',
       'A-108', 'A-109', 'A-110', 'A-111', 'A-112', 'A-113', 'A-114',
       'A-115', 'A-116', 'A-117', 'A-118', 'A-119', 'A-120', 'A-121',
       'A-122', 'A-123', 'A-124', 'A-125', 'A-126', 'A-127']

channel_list_B = ['B-000', 'B-001', 'B-002',
       'B-003', 'B-004', 'B-005', 'B-006', 'B-007', 'B-008', 'B-009',
       'B-010', 'B-011', 'B-012', 'B-013', 'B-014', 'B-015', 'B-016',
       'B-017', 'B-018', 'B-019', 'B-020', 'B-021', 'B-022', 'B-023',
       'B-024', 'B-025', 'B-026', 'B-027', 'B-028', 'B-029', 'B-030',
       'B-031', 'B-032', 'B-033', 'B-034', 'B-035', 'B-036', 'B-037',
       'B-038', 'B-039', 'B-040', 'B-041', 'B-042', 'B-043', 'B-044',
       'B-045', 'B-046', 'B-047', 'B-048', 'B-049', 'B-050', 'B-051',
       'B-052', 'B-053', 'B-054', 'B-055', 'B-056', 'B-057', 'B-058',
       'B-059', 'B-060', 'B-061', 'B-062', 'B-063', 'B-064', 'B-065',
       'B-066', 'B-067', 'B-068', 'B-069', 'B-070', 'B-071', 'B-072',
       'B-073', 'B-074', 'B-075', 'B-076', 'B-077', 'B-078', 'B-079',
       'B-080', 'B-081', 'B-082', 'B-083', 'B-084', 'B-085', 'B-086',
       'B-087', 'B-088', 'B-089', 'B-090', 'B-091', 'B-092', 'B-093',
       'B-094', 'B-095', 'B-096', 'B-097', 'B-098', 'B-099', 'B-100',
       'B-101', 'B-102', 'B-103', 'B-104', 'B-105', 'B-106', 'B-107',
       'B-108', 'B-109', 'B-110', 'B-111', 'B-112', 'B-113', 'B-114',
       'B-115', 'B-116', 'B-117', 'B-118', 'B-119', 'B-120', 'B-121',
       'B-122', 'B-123', 'B-124', 'B-125', 'B-126', 'B-127']

In [ ]:
# 批量处理所有数据
for mouse_name, dates_dict in file_dict.items():
    for date, data_path in dates_dict.items():
        print(f"\n{'='*60}")
        print(f"处理: {mouse_name}, 日期: {date}, 路径: {data_path}")
        print(f"{'='*60}")
        
        # 获取该数据路径下的所有rhd文件
        file_list_path = Path(data_path)
        rhd_files = list(file_list_path.glob("*.rhd"))
        file_list = sorted(rhd_files)
        
        if len(file_list) == 0:
            print(f"警告: 在 {data_path} 中未找到.rhd文件，跳过")
            continue
        
        # 读取并合并所有rhd文件
        recording_raw_list = []
        for file in file_list:
            recording_raw_list.append(se.read_intan(file, stream_id='0'))
        recording_raw = concatenate_recordings(recording_list=recording_raw_list)
        
        # 检测通道类型并选择对应的channel_list
        available_channels = recording_raw.get_channel_ids()
        if 'A-127' in available_channels:
            channel_list = channel_list_A
            print(f"检测到A通道，使用channel_list_A")
        elif 'B-127' in available_channels:
            channel_list = channel_list_B
            print(f"检测到B通道，使用channel_list_B")
        else:
            print(f"警告: 未找到A-127或B-127通道，可用通道: {available_channels[:10]}...")
            print(f"跳过此数据")
            continue
        
        # 设置输出文件夹路径
        output_folder = f'/media/ubuntu/sda/mouse_test/sorted/mountainsort/{mouse_name}/{date}'
        os.makedirs(output_folder, exist_ok=True)
        
        print(f"输出文件夹: {output_folder}")
        
        # 选择通道
        recording_raw = recording_raw.select_channels(channel_list)

        # 预处理
        recording_raw = spre.unsigned_to_signed(recording_raw)
        recording_raw = spre.resample(recording_raw, 10000)
        recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
        recording_recorded = spre.notch_filter(recording_recorded, freq=50)
        recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

        probe = read_probeinterface('/media/ubuntu/sda/mouse_test/probe/tip_probe_128_1.json')
        recording_f = recording_f.set_probegroup(probe)

        # 保存预处理后的数据
        recording_preprocessed = recording_f.save(format="binary", n_jobs = 20)

        default_params = {
                'detect_sign': -1,  # Use -1, 0, or 1, depending on the sign of the spikes in the recording
                'adjacency_radius': 120,  # Use -1 to include all channels in every neighborhood
                'freq_min': 300,  # Use None for no bandpass filtering
                'freq_max': 3000,
                'filter': True,
                'whiten': True,  # Whether to do channel whitening as part of preprocessing
                'num_workers': 20,
                'clip_size': 50,
                'detect_threshold': 4, # 5
                'detect_interval': 3,  # Minimum number of timepoints between events detected on the same channel, 30
            }
        # 运行Kilosort4排序
        sorting_mountainsort = ss.run_sorter(sorter_name='mountainsort4',
                                     recording=recording_preprocessed,
                                     remove_existing_folder='True',
                                     folder=output_folder,
                                     **default_params)

        # 创建排序分析器
        analyzer_mountainsort = si.create_sorting_analyzer(
            sorting=sorting_mountainsort, 
            recording=recording_preprocessed, 
            format='binary_folder', 
            folder=output_folder + '/analyzer_kilosort4_binary'
        )

        # 计算扩展信息
        extensions_to_compute = [
            "random_spikes",
            "waveforms",
            "noise_levels",
            "templates",
            "unit_locations",
            "spike_locations",
            "correlograms",
            "template_similarity"
        ]

        extension_params = {
            "unit_locations": {"method": "center_of_mass"},
            "spike_locations": {"ms_before": 0.1},
            "correlograms": {"bin_ms": 0.1},
            "template_similarity": {"method": "cosine_similarity"}
        }

        analyzer_mountainsort.compute(extensions_to_compute, extension_params=extension_params, n_jobs = 20)

        # 读取spikes.npy并检查无效的spike
        spikes_path = output_folder + "/analyzer_kilosort4_binary/sorting/spikes.npy"
        spikes = np.load(spikes_path)

        # 获取recording的总样本数
        total_samples = recording_f.get_num_samples()

        # 检查第一个和最后一个spike
        first_spike_valid = spikes[0]['sample_index'] >= 0
        last_spike_valid = spikes[-1]['sample_index'] < total_samples

        # 如果第一个或最后一个spike无效，删除所有无效的spike
        if not first_spike_valid or not last_spike_valid:
            # 创建有效spike的掩码：sample_index >= 0 且 < total_samples
            valid_mask = (spikes['sample_index'] >= 0) & (spikes['sample_index'] < total_samples)
            spikes_filtered = spikes[valid_mask]
            
            # 保存过滤后的spikes
            np.save(spikes_path, spikes_filtered)
            print(f"删除了 {len(spikes) - len(spikes_filtered)} 个无效的spike")
            print(f"原始spike数量: {len(spikes)}, 过滤后: {len(spikes_filtered)}")
        else:
            print("所有spike都在有效范围内")

        qm_params = sqm.get_default_qm_params()
        analyzer_mountainsort.compute("quality_metrics", qm_params, n_jobs = 20)

        # 导出到phy格式
        sexp.export_to_phy(analyzer_mountainsort, output_folder + "/phy_folder_for_kilosort", verbose=True, n_jobs = 20)
        
        print(f"完成处理: {mouse_name}, 日期: {date}\n")

print("\n所有数据处理完成！")

In [ ]:
# Cell 5的内容已合并到Cell 4中


In [ ]:

# # 修正phy文件夹中的spike_times.npy, spike_clusters.npy和spike_templates.npy
# phy_folder = output_folder + "/phy_folder_for_kilosort"
# spike_times_path = phy_folder + "/spike_times.npy"
# spike_clusters_path = phy_folder + "/spike_clusters.npy"
# spike_templates_path = phy_folder + "/spike_templates.npy"

# if os.path.exists(spike_times_path):
#     spike_times = np.load(spike_times_path)
#     # spike_times可能是2D数组，需要展平来检查第一项
#     spike_times_flat = spike_times.flatten() if spike_times.ndim > 1 else spike_times
#     original_spike_times_len = len(spike_times)
    
#     # 检查第一项是否小于0
#     if len(spike_times_flat) > 0 and spike_times_flat[0] < 0:
#         first_value = spike_times_flat[0]
#         print(f"发现spike_times第一项小于0: {first_value}")
        
#         # 删除第一项（保持原始维度）
#         if spike_times.ndim == 2:
#             spike_times_filtered = spike_times[1:, :]
#         else:
#             spike_times_filtered = spike_times[1:]
#         np.save(spike_times_path, spike_times_filtered)
#         print(f"已删除spike_times的第一项，从 {original_spike_times_len} 减少到 {len(spike_times_filtered)}")
        
#         # 检查spike_clusters.npy和spike_templates.npy的维度是否与修正前的spike_times匹配
#         if os.path.exists(spike_clusters_path):
#             spike_clusters = np.load(spike_clusters_path)
#             # spike_clusters也可能是2D数组
#             spike_clusters_flat = spike_clusters.flatten() if spike_clusters.ndim > 1 else spike_clusters
#             if len(spike_clusters_flat) == original_spike_times_len:
#                 # 删除第一项（保持原始维度）
#                 if spike_clusters.ndim == 2:
#                     spike_clusters_filtered = spike_clusters[1:, :]
#                 else:
#                     spike_clusters_filtered = spike_clusters[1:]
#                 np.save(spike_clusters_path, spike_clusters_filtered)
#                 print(f"已删除spike_clusters的第一项，从 {len(spike_clusters)} 减少到 {len(spike_clusters_filtered)}")
#             else:
#                 print(f"spike_clusters维度({len(spike_clusters_flat)})与修正前的spike_times({original_spike_times_len})不匹配，跳过")
        
#         if os.path.exists(spike_templates_path):
#             spike_templates = np.load(spike_templates_path)
#             # spike_templates也可能是2D数组
#             spike_templates_flat = spike_templates.flatten() if spike_templates.ndim > 1 else spike_templates
#             if len(spike_templates_flat) == original_spike_times_len:
#                 # 删除第一项（保持原始维度）
#                 if spike_templates.ndim == 2:
#                     spike_templates_filtered = spike_templates[1:, :]
#                 else:
#                     spike_templates_filtered = spike_templates[1:]
#                 np.save(spike_templates_path, spike_templates_filtered)
#                 print(f"已删除spike_templates的第一项，从 {len(spike_templates)} 减少到 {len(spike_templates_filtered)}")
#             else:
#                 print(f"spike_templates维度({len(spike_templates_flat)})与修正前的spike_times({original_spike_times_len})不匹配，跳过")
#     else:
#         first_value = spike_times_flat[0] if len(spike_times_flat) > 0 else 'empty'
#         print(f"spike_times第一项检查通过: {first_value}")
# else:
#     print(f"警告: 未找到spike_times.npy文件: {spike_times_path}")